"""Phase 2 — RAG sur la documentation Python officielle (PyTutor).

Concepts LangChain travaillés :
  1. DocumentLoader : aspirer des pages web (WebBaseLoader)
  2. TextSplitter : découper en chunks avec chevauchement
  3. Embeddings : encoder le texte en vecteurs (HuggingFace, local)
  4. VectorStore : indexer et persister sur disque (Chroma)
  5. Retriever : interface Runnable pour la recherche par similarité
  6. Chaîne RAG complète en LCEL
  7. Récupérer la réponse ET les sources (RunnableParallel)
  8. Comparaison similarity vs MMR (stretch)

### ARCHITECTURE DU SCRIPT
----------------------
On sépare l'indexation (lente, faite une fois) de l'interrogation (rapide,
faite à chaque question). La fonction `build_or_load_vectorstore()` réindexe
seulement si le dossier `chroma_db/` n'existe pas. Donc :
  - 1er lancement : ~30s d'indexation (téléchargement embeddings + scraping)
  - Lancements suivants : <1s (rechargement depuis disque)

Lance ce fichier directement pour voir la démo complète.
"""

In [ ]:
from __future__ import annotations

from pathlib import Path

import requests
from bs4 import BeautifulSoup

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

In [ ]:
import os
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate

from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

HAIKU = "claude-haiku-4-5-20251001"
SONNET = "claude-sonnet-4-6"
DEFAULT_MODEL = HAIKU
load_dotenv()


def get_model(model_name: str = DEFAULT_MODEL, temperature: float = 0.0, max_tokens: int = 2048) -> ChatAnthropic:
    """Renvoie une instance configurée de ChatAnthropic.
    Lève une erreur claire si la clé API n'est pas trouvée.
    """
    if not os.getenv("ANTHROPIC_API_KEY"):
        raise RuntimeError(
            "ANTHROPIC_API_KEY introuvable. "
            "As-tu créé un fichier .env à partir de .env ?"
        )
    return ChatAnthropic(model=model_name, temperature=temperature, max_tokens=max_tokens)


<H2><font color="#b22222"> Sources : pages de la doc officielle Python à indexer </font> </H2>

Choix éditorial : on couvre les sujets qui reviennent le plus en cours.
Tu peux ajouter/retirer librement, l'indexation est incrémentale en théorie
(en pratique on rebuild tout pour rester simple dans ce TP).


In [9]:
PYTHON_DOCS_URLS = [
    "https://docs.python.org/3/tutorial/controlflow.html",
    "https://docs.python.org/3/tutorial/datastructures.html",
    "https://docs.python.org/3/tutorial/classes.html",
    "https://docs.python.org/3/tutorial/errors.html",
    "https://docs.python.org/3/tutorial/modules.html",
    "https://docs.python.org/3/howto/functional.html",
    "https://docs.python.org/3/glossary.html",
]

PERSIST_DIR = "./chroma_db"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"  # 384 dim, rapide, multilingue
COLLECTION_NAME = "python_docs"


<H2><font color="#b22222"> 2. Indexation : loader -> splitter -> embeddings -> Chroma </font> </H2>

Si la base existe déjà sur disque, on la recharge sans rien refaire.
Sinon on aspire les URL, on découpe, on encode, on persiste.

In [ ]:
USER_AGENT = "PyTutor-Edu/0.1"


def load_web_pages(urls: list[str]) -> list[Document]:
    """Télécharge chaque URL et renvoie un `Document` par page.

    Remplace `langchain_community.document_loaders.WebBaseLoader` (sunset
    en LangChain 1.x) par un équivalent minimal : `requests` pour la requête
    HTTP, `BeautifulSoup` pour extraire le texte brut. C'est exactement ce
    que faisait WebBaseLoader en interne.
    """
    docs: list[Document] = []
    headers = {"User-Agent": USER_AGENT}
    for url in urls:
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()
        # On passe les bytes bruts (response.content) plutôt que response.text :
        # requests devine parfois mal l'encoding à partir des headers HTTP
        # (Latin-1 par défaut pour HTML), alors que BeautifulSoup détecte
        # correctement l'UTF-8 à partir du <meta charset> de la page.
        soup = BeautifulSoup(response.content, "lxml")
        text = soup.get_text(separator="\n", strip=True)
        docs.append(Document(page_content=text, metadata={"source": url}))
    return docs


def get_embeddings() -> HuggingFaceEmbeddings:
    """Singleton de fait : un seul modèle d'embeddings réutilisé partout."""
    return HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)


def build_or_load_vectorstore(force_rebuild: bool = False) -> Chroma:
    """Renvoie le vectorstore prêt à l'emploi.
    - Si `chroma_db/` existe et `force_rebuild=False`, on recharge depuis disque.
    - Sinon, on aspire toutes les URL et on réindexe.
    """
    embeddings = get_embeddings()
    persist_path = Path(PERSIST_DIR)

    if persist_path.exists() and not force_rebuild:
        print(f"✓ Rechargement de l'index depuis {PERSIST_DIR}")
        return Chroma(
            collection_name=COLLECTION_NAME,
            persist_directory=PERSIST_DIR,
            embedding_function=embeddings,
        )

    print(f"→ Indexation de {len(PYTHON_DOCS_URLS)} pages de docs.python.org...")

    # --- Chargement : une page = un Document, URL en metadata. ---
    raw_docs = load_web_pages(PYTHON_DOCS_URLS)
    print(f"  {len(raw_docs)} pages chargées")

    # --- Découpage en chunks ---
    # chunk_size en CARACTÈRES (pas tokens). 1000 ≈ 200-250 tokens : un bon
    # compromis entre granularité (recherche précise) et contexte (chunk
    # auto-suffisant pour répondre).
    # chunk_overlap = 200 : 20% de chevauchement, on évite de couper une idée.
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        # L'ordre des séparateurs compte : on essaie d'abord de couper aux
        # endroits naturels, et seulement en dernier recours au milieu d'un mot.
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = splitter.split_documents(raw_docs)
    print(f"  {len(chunks)} chunks générés")

    # --- Indexation + persistance ---
    # Chroma.from_documents calcule les embeddings et persiste en une étape.
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=COLLECTION_NAME,
        persist_directory=PERSIST_DIR,
    )
    print(f"✓ Index créé et persisté dans {PERSIST_DIR}\n")
    return vectorstore


<H2><font color="#b22222"> 3. La chaîne RAG en LCEL </font> </H2>

### Pattern canonique :
<pre>
   {"context": retriever | format, "question": passthrough}
       | prompt | model | parser
</pre>
### Lis ce code en regardant CE QU'UN INPUT TRAVERSE :
   1. l'utilisateur passe une string : "Comment fonctionnent les générateurs ?"
   2. le dict en entrée crée DEUX branches parallèles :
        - branche "context" : la string -> retriever -> liste de docs -> format -> string
        - branche "question" : la string -> passthrough -> string (inchangée)
   3. le prompt reçoit {"context": "...", "question": "..."} et formate
   4. le modèle répond
   5. le parser extrait le texte

In [11]:
RAG_PROMPT = ChatPromptTemplate.from_messages([
    (
        "system",
        "Tu es un assistant pédagogique pour des élèves de Python. "
        "Réponds à la question en t'appuyant UNIQUEMENT sur le contexte fourni.\n"
        "Si la réponse n'est pas dans le contexte, dis-le explicitement plutôt "
        "que d'inventer. Cite le ou les chunks utilisés par leur numéro entre "
        "crochets, par exemple [1] ou [2,3]."
    ),
    (
        "human",
        "Contexte :\n{context}\n\n"
        "Question : {question}\n\n"
        "Réponse pédagogique :"
    ),
])


def format_docs(docs: list[Document]) -> str:
    """Concatène les documents retrouvés en un seul bloc de contexte numéroté.

    Le numéro [N] permet au modèle de citer ses sources.
    """
    return "\n\n".join(
        f"[{i + 1}] (source: {doc.metadata.get('source', '?')})\n{doc.page_content}"
        for i, doc in enumerate(docs)
    )


def build_rag_chain(retriever):
    """Chaîne RAG simple : input string -> output string (la réponse)."""
    return (
        {
            "context": retriever | format_docs,
            "question": RunnablePassthrough(),
        }
        | RAG_PROMPT
        | get_model()
        | StrOutputParser()
    )


def build_rag_chain_with_sources(retriever):
    """Chaîne RAG qui renvoie un dict {answer, sources}.

    Pattern intéressant : on utilise RunnableParallel pour FAIRE DEUX CHOSES
    en même temps : générer la réponse ET garder les docs retrouvés pour
    inspection. C'est le moyen idiomatique de récupérer les sources avec LCEL.
    """
    # Étape 1 : on récupère les docs UNE FOIS et on les passe en aval.
    retrieve = RunnableParallel(
        context=retriever,
        question=RunnablePassthrough(),
    )

    # Étape 2 : à partir de {context: [Doc...], question: str}, on construit
    # la réponse en formatant les docs, puis on remet tout dans un dict final.
    answer_chain = (
        RunnablePassthrough.assign(
            context=lambda x: format_docs(x["context"]),
        )
        | RAG_PROMPT
        | get_model()
        | StrOutputParser()
    )

    # `.assign(answer=...)` ajoute la clé "answer" au dict courant, sans
    # toucher aux autres clés. À la fin on a {context: [Doc...], question, answer}.
    return retrieve | RunnablePassthrough.assign(answer=answer_chain)


<H2><font color="#b22222"> 4. Démo : 3 questions, deux stratégies de retrieval </font> </H2>


In [12]:
DEMO_QUESTIONS = [
    "Quelle est la différence entre une liste et un tuple ?",
    "Comment écrire un générateur en Python ?",
    "À quoi sert le mot-clé `yield` exactement ?",
]



# --- Indexation (ou rechargement) ---
vectorstore = build_or_load_vectorstore()

# --- Démo 1 : RAG simple, sortie texte ---
print("=" * 70)
print("DÉMO 1 — RAG simple (retriever similarity, k=4)")
print("=" * 70)
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)
chain = build_rag_chain(retriever)

question = DEMO_QUESTIONS[1]
print(f"\nQuestion : {question}\n")
answer = chain.invoke(question)
print(answer)

# --- Démo 2 : RAG avec sources ---
print("\n" + "=" * 70)
print("DÉMO 2 — RAG avec inspection des sources")
print("=" * 70)
chain_with_sources = build_rag_chain_with_sources(retriever)
question = DEMO_QUESTIONS[2]
print(f"\nQuestion : {question}\n")
result = chain_with_sources.invoke(question)
print("RÉPONSE :")
print(result["answer"])
print("\nSOURCES UTILISÉES :")
for i, doc in enumerate(result["context"], 1):
    snippet = doc.page_content[:120].replace("\n", " ")
    print(f"  [{i}] {doc.metadata.get('source', '?')}")
    print(f"      {snippet}...")

# --- Démo 3 : comparaison similarity vs MMR ---
print("\n" + "=" * 70)
print("DÉMO 3 — Similarity vs MMR sur la MÊME question")
print("=" * 70)
question = DEMO_QUESTIONS[0]
print(f"\nQuestion : {question}")

sim_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    # fetch_k : on récupère 20 candidats avant de sélectionner les 3 plus diversifiés.
    # lambda_mult : 1.0 = pure similarité, 0.0 = pure diversité.
    search_kwargs={"k": 3, "fetch_k": 20, "lambda_mult": 0.5},
)

print("\n--- Top-3 similarity ---")
for i, doc in enumerate(sim_retriever.invoke(question), 1):
    print(f"  [{i}] {doc.page_content[:100].replace(chr(10), ' ')}...")

print("\n--- Top-3 MMR (diversifié) ---")
for i, doc in enumerate(mmr_retriever.invoke(question), 1):
    print(f"  [{i}] {doc.page_content[:100].replace(chr(10), ' ')}...")

print("\n" + "=" * 70)
print("Observe : MMR évite les doublons sémantiques que retourne la")
print("similarity pure. Pour une question généraliste comme 'liste vs")
print("tuple', MMR ramène souvent des chunks plus complémentaires.")
print("=" * 70)


✓ Rechargement de l'index depuis ./chroma_db
DÉMO 1 — RAG simple (retriever similarity, k=4)

Question : Comment écrire un générateur en Python ?

# Comment écrire un générateur en Python ?

Un générateur en Python est une fonction spéciale qui utilise le mot-clé **`yield`** [1].

## Caractéristiques principales :

1. **Définition** : Toute fonction contenant le mot-clé `yield` est une fonction générateur. Python la compile de manière spéciale [1].

2. **Comportement** : 
   - Au lieu de retourner une seule valeur avec `return`, un générateur retourne un **objet générateur** qui supporte le protocole itérateur [1]
   - Quand on appelle `yield`, le générateur produit une valeur et **suspend son exécution** [1]
   - Les variables locales sont **préservées** entre les appels [1]

3. **Utilisation** : On utilise la fonction `next()` pour récupérer les valeurs successives du générateur [1]

## Exemple simple :

```python
def generate_ints(n):
    i = 0
    while i < n:
        yield i
     